In [4]:
!pip install gradio opencv-python-headless -q
!wget -q -O yunet.onnx https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx

import cv2
import gradio as gr

detector = cv2.FaceDetectorYN.create(
    "yunet.onnx", "", (320, 320),
    score_threshold=0.6,   # lower = finds more faces, higher = fewer false boxes
    nms_threshold=0.3,
    top_k=5000,
)

def detect(img):
    img = img.copy()
    # shrink very large photos for speed
    h, w = img.shape[:2]
    if w > 1280:
        scale = 1280 / w
        img = cv2.resize(img, (1280, int(h * scale)))
        h, w = img.shape[:2]

    bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    detector.setInputSize((w, h))
    _, faces = detector.detect(bgr)

    count = 0
    if faces is not None:
        count = len(faces)
        for f in faces:
            x, y, bw, bh = f[:4].astype(int)
            cv2.rectangle(img, (x, y), (x + bw, y + bh), (0, 255, 0), 3)
            cv2.putText(img, f"{f[14]:.2f}", (x, max(y - 8, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            for i in range(5):  # eyes, nose, mouth corners
                px, py = int(f[4 + 2 * i]), int(f[5 + 2 * i])
                cv2.circle(img, (px, py), 3, (255, 0, 0), -1)
    return img, f"Faces found: {count}"

app = gr.Interface(
    fn=detect,
    inputs=gr.Image(type="numpy", sources=["upload", "webcam"], label="Upload a photo"),
    outputs=[gr.Image(label="Result"), gr.Text(label="Count")],
    title="Face Detection App (YuNet)",
)
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ac6525d29c631967bc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
